In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Consolidated_Pipeline/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'fmcg', "Catalog")
dbutils.widgets.text('data_source', 'customers', "Data Source")

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f"fmcg.default.{data_source}"
print(base_path)

In [0]:
df = (
    spark.read.table(base_path)
    .withColumn("read_timestamp",F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
    )
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

## Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count")>1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(['customer_id'])

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn('customer_name', F.trim(F.col("customer_name")))

In [0]:
df_silver.select("city").distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'NewDelhee': 'New Delhi',
    'NewDelhi': 'New Delhi',
    'New Dheli': 'New Delhi',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad'
}

allowed = ['Bengaluru', 'New Delhi', 'Hyderabad']

df_silver = (
    df_silver
    .replace(city_mapping, subset='city')
    .withColumn('city',
                F.when(F.col('city').isNull(), None)
                .when(F.col('city').isin(allowed), F.col('city'))
                .otherwise(None)
                )
)

display(df_silver.limit(10))

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col('customer_name').isNull(), None)
    .otherwise(F.initcap("customer_name"))
)

display(df_silver)

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_name = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

df_silver.filter(F.col("customer_name").isin(null_customer_name)).show(truncate=False)

In [0]:
customer_city_fix = {
    789403: 'New Delhi',

    789420: 'Bengaluru',

    789521: 'Hyderabad',
    789522: 'New Delhi',

    789603: 'Hyderabad'
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id","fixed_city"]
)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        'city',
        F.coalesce('city', 'fixed_city')
    )
    .drop('fixed_city')
)

display(df_silver)

In [0]:
df_silver = df_silver.withColumn('customer_id', F.col("customer_id").cast('string'))

df_silver.printSchema()

In [0]:
df_silver = (
    df_silver
    .withColumn('customer',
                F.concat_ws("-", F.col('customer_name'), F.coalesce(F.col("city"), F.lit('Unknown')))
                )
    
    .withColumn("market", F.lit('India'))
    .withColumn("platform", F.lit('Sports Bar'))
    .withColumn("channel", F.lit('Acquisition'))
)

display(df_silver.limit(10))

In [0]:
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

## Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

display(df_gold)

In [0]:
df_gold.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

## Merging Part

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
    )

In [0]:
#UPSURT Operation
delta_table.alias('target').merge(
    source = df_child_customers.alias('source'),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

